<a href="https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/VIX_FINAL_TFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VIX Final TFT v1 — TFT sur tous les horizons × régimes, résumable (3/4)

**Rôle.** Troisième des 4 notebooks de la campagne finale. Recharge le dataset produit par `VIX_FINAL_FEATURES` et entraîne le **TFT** (architecture reprise verbatim de `VIX_ML3`) sur la grille complète : 6 horizons × 4 régimes × 5 folds walk-forward = 120 modèles.

**Contexte** : en walk-forward, le TFT s'était déjà effondré à h=5j GLOBAL (`VIX_TFT_WF` : F1_UP_FORT 0.588→0.118). Ce notebook vérifie si un autre horizon ou régime s'en sort mieux — et si les nouvelles features (interactions inter-tickers) changent la donne.

**Méthodologie reprise** : pas de SMOTE avant le fenêtrage (les fenêtres de 21 jours restent de vraies séquences chronologiques ; déséquilibre géré par la Focal Loss pondérée), hold-out chronologique réel pour l'early stopping. **Nouveau** : pour les régimes non-GLOBAL (jours dispersés dans le calendrier), la fenêtre de 21 jours porte sur les 21 jours *du régime* les plus récents, pas 21 jours calendaires consécutifs — seule option cohérente puisque les jours CALM/STRESS ne sont pas contigus.

**Résumable** comme les autres notebooks de la série : reprise automatique + checkpoints périodiques (moins critique ici que pour le scan ML — 120 modèles est beaucoup plus rapide que 66 000 fits ML, mais chaque TFT prend plusieurs minutes, donc ça reste utile).

**GPU fortement recommandé** (`Exécution → Modifier le type d'exécution → GPU`).


In [ ]:
import subprocess, sys
pkgs = ['xlsxwriter','pyarrow','torch']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


In [ ]:
import os, time, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, accuracy_score

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

NOTEBOOK_NAME = 'VIX_FINAL_TFT'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'flat_thr': 0.003,
    'horizons': [1, 2, 3, 5, 7, 10],
    'regimes': ['GLOBAL', 'CALM', 'NORMAL', 'STRESS'],
    'n_wf_folds': 5,
    'min_train_frac': 0.40,   # doit matcher VIX_FINAL_FEATURES
    'dl_lookback': 21, 'dl_epochs': 50, 'dl_batch': 64, 'dl_lr': 3e-4, 'dl_dropout': 0.3,
    'dl_holdout_frac': 0.15,
    'min_train_rows': 200, 'min_test_rows': 20,
}
TARGET_COL = 'VIX_Amplitude_Class'
RESULTS_CSV = 'vix_final_tft_results.csv'
GITHUB_REPO = 'LP-D/claude'
FEATURES_BRANCH = 'results/vix-final-features'
RESULTS_BRANCH = 'results/vix-final-tft'

# Features DL — reprises verbatim de VIX_ML3 (jeu validé pour le TFT)
DL_FEATURE_SET = [
    'P_stress_HMM', 'VIX_Residual', 'VIX_Innovation',
    'egarch_condvar', 'egarch_delta', 'gjr_condvar',
    'heston_xi', 'heston_theta', 'heston_kappa', 'heston_v0',
    'heston_rho', 'heston_feller', 'heston_v0_minus_theta',
    'heston_xi_zscore', 'heston_kappa_zscore',
    'VRP', 'VRP_zscore', 'VRP_ma5',
    'jump_intensity_20d', 'jump_intensity_60d',
    'hawkes_intensity', 'hawkes_zscore',
    'impl_corr_proxy',
    'vix_vol_of_vol_5d', 'vix_vol_of_vol_10d',
    'vix_zscore_5d', 'vix_zscore_10d',
    'vix_momentum_3d', 'vix_acceleration_1d', 'vix_acceleration_3d',
    'vix_erratic_ratio', 'vix_vol_ratio_5_60',
    'vix_max_abs_ret_5d', 'vix_spx_corr_30d',
    'spx_drawdown_252d', 'spx_vol_5d',
    'IDX_VIX_ret_1d', 'IDX_VIX_ret_5d', 'IDX_VIX_ret_20d',
    'IDX_VIX_vol_20d', 'IDX_VIX_zscore_60d',
]

n_combos = len(CONFIG['horizons']) * len(CONFIG['regimes']) * CONFIG['n_wf_folds']
print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | device={device} | grille = {n_combos} modèles TFT "
      f"({len(CONFIG['horizons'])}h × {len(CONFIG['regimes'])}reg × {CONFIG['n_wf_folds']}folds)")


In [ ]:
# ============================================================
# CHARGEMENT DU DATASET PARTAGÉ (produit par VIX_FINAL_FEATURES)
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

if not os.path.exists('vix_final_features.parquet'):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_REPO}.git"
    workdir = "/content/_vix_features_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", FEATURES_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(
            "Impossible de récupérer le dataset partagé depuis "
            f"'{FEATURES_BRANCH}'. As-tu bien exécuté VIX_FINAL_FEATURES.ipynb en premier "
            f"(et poussé son résultat) ? Détail: {clone.stderr[-500:]}")
    subprocess.run(["cp", f"{workdir}/vix_final_features.parquet", "."], check=True)
    subprocess.run(["cp", f"{workdir}/vix_final_features_meta.json", "."], check=True)
    print(f"[PULL OK] Dataset récupéré depuis '{FEATURES_BRANCH}'")
else:
    print("[SKIP] vix_final_features.parquet déjà présent localement")

df_features = pd.read_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json') as f:
    meta = json.load(f)
FEATURE_POOL = meta['feature_pool']
VIX_COL = meta['vix_col']; SPX_COL = meta['spx_col']
print(f"Dataset: {df_features.shape} | VIX={VIX_COL} | pool: {len(FEATURE_POOL)} features "
      f"(dont {len(meta['interaction_features'])} interactions) | source: {meta['date_min']} → {meta['date_max']}")

all_dates = df_features.dropna(how='all').index.sort_values()
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
for k in range(CONFIG['n_wf_folds']):
    print(f"  Fold {k+1}: train → {all_dates[FOLD_CUTS[k]-1].date()} | "
          f"test {all_dates[FOLD_CUTS[k]].date()} → {all_dates[FOLD_CUTS[k+1]-1].date()}")


In [ ]:
def build_target(vix_series, horizon, split_idx):
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def metrics(y_true, y_pred):
    dm = {0: 'DOWN', 1: 'DOWN', 2: 'UP', 3: 'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {'F1_4cls': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
         'Acc_dir': round(accuracy_score(yd_t, yd_p), 4),
         'F1_dir': round(f1_score(yd_t, yd_p, average='macro', zero_division=0), 4)}
    ui = [i for i, y in enumerate(y_true) if dm[y] == 'UP']
    di = [i for i, y in enumerate(y_true) if dm[y] == 'DOWN']
    if len(ui) >= 10:
        yt = ['FORT' if y_true[i] == 3 else 'FAIBLE' for i in ui]
        yp = ['FORT' if y_pred[i] == 3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_UP_FORT'] = np.nan
    if len(di) >= 10:
        yt = ['FORT' if y_true[i] == 0 else 'FAIBLE' for i in di]
        yp = ['FORT' if y_pred[i] == 0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_DOWN_FORT'] = np.nan
    return m

print("Helpers OK (build_target, metrics)")


## Architectures Deep Learning — principes et fondements

La cellule suivante définit 6 architectures de séquences (entrée : fenêtre glissante des `dl_lookback` derniers jours de features) plus la fonction de perte et les boucles d'entraînement/évaluation.

- **LSTM** (Long Short-Term Memory, Hochreiter & Schmidhuber, 1997) — réseau récurrent avec portes (entrée/oubli/sortie) qui régulent la circulation d'un état de cellule $c_t$ le long de la séquence, ce qui atténue le problème de gradient qui s'évanouit des RNN classiques et permet de capter des dépendances de moyen terme.
- **TCN** (Temporal Convolutional Network) — convolutions **causales dilatées** empilées (dilatation $2^i$ à la couche $i$) : chaque sortie ne dépend que du passé, et le champ réceptif croît exponentiellement avec la profondeur sans récurrence — parallélisable, contrairement au LSTM.
- **Transformer** (Vaswani et al., 2017) — auto-attention : chaque pas de temps calcule une moyenne pondérée de tous les autres pas via des poids $\text{softmax}(QK^\top/\sqrt{d})$ appris, ce qui capte des dépendances à longue portée sans la contrainte séquentielle de la récurrence. Un encodage positionnel (`nn.Embedding` sur l'index temporel ici) est nécessaire car l'attention est intrinsèquement invariante à l'ordre.
- **CNN-LSTM** — une couche convolutive 1D extrait des motifs locaux à court terme, puis un LSTM modélise les dépendances temporelles sur la séquence de features convoluées.
- **TFT** (Temporal Fusion Transformer, simplifié — Lim et al., 2019) — combine une porte de sélection de variables (`vsn`, un mécanisme d'attention sur les features en entrée), un LSTM, une couche d'auto-attention, et une **Gated Linear Unit** ($\text{GLU}(x)=(W_1x)\odot\sigma(W_2x)$) qui permet au réseau de choisir dynamiquement de laisser passer ou non chaque composante.
- **Mamba** (State Space Model, simplifié — Gu & Dao, 2023) — remplace l'attention par une récurrence linéaire à noyau de convolution appris (ici approximée par une convolution causale à noyau `softmax`), avec un coût linéaire en longueur de séquence (contre quadratique pour l'attention classique).

### Focal Loss
**Principe** — Variante de l'entropie croisée qui pondère chaque exemple par $(1-p_t)^\gamma$ (Lin et al., 2017) : les exemples déjà bien classés ($p_t$ proche de 1) contribuent peu au gradient, ce qui force le réseau à se concentrer sur les exemples difficiles/minoritaires — complémentaire au rééquilibrage SMOTE pour gérer le déséquilibre des classes. Le paramètre `alpha` (ici `compute_cw`, poids inverses de fréquence) pondère en plus chaque classe, et `ls` applique un *label smoothing* (cible non one-hot, $1-\varepsilon$ sur la vraie classe et $\varepsilon/n$ sur les autres) pour limiter la sur-confiance du modèle.


In [ ]:
# ============================================================
# ARCHITECTURE DL — TFT uniquement (reprise verbatim de VIX_ML3, non modifiée :
# c'est précisément le modèle dont le rapport donne F1_UP_FORT=0.5879 en
# split statique h=5j — on teste ici si ce score survit au walk-forward)
# ============================================================
class PreScaledDS(torch.utils.data.Dataset):
    def __init__(self,X,y,lb=CONFIG['dl_lookback']):
        self.X=np.nan_to_num(X.astype(np.float32),nan=0.,posinf=0.,neginf=0.)
        self.y=y.astype(np.int64); self.lb=lb
    def __len__(self): return max(0,len(self.X)-self.lb)
    def __getitem__(self,i):
        return torch.tensor(self.X[i:i+self.lb]),torch.tensor(self.y[i+self.lb])

class FocalLoss(nn.Module):
    def __init__(self,gamma=2.,alpha=None,ls=0.1):
        super().__init__(); self.g=gamma; self.a=alpha; self.ls=ls
    def forward(self,logits,targets):
        n=logits.size(1)
        oh=torch.zeros_like(logits).scatter_(1,targets.unsqueeze(1),1)
        smooth=oh*(1-self.ls)+self.ls/n
        lp=torch.log_softmax(logits,dim=1); p=lp.exp()
        at=self.a.to(logits.device)[targets].unsqueeze(1) if self.a is not None else 1.
        return (-(at*(1-p)**self.g*smooth*lp).sum(dim=1)).mean()

class GLU(nn.Module):
    def __init__(self,d): super().__init__(); self.f=nn.Linear(d,d); self.g=nn.Linear(d,d)
    def forward(self,x): return self.f(x)*torch.sigmoid(self.g(x))

class VIX_TFT(nn.Module):
    def __init__(self,d,dm=128,nh=4,nl=2,drop=CONFIG['dl_dropout'],nc=4):
        super().__init__()
        self.vsn=nn.Sequential(nn.Linear(d,dm),nn.GELU(),nn.Linear(dm,d),nn.Softmax(dim=-1))
        self.proj=nn.Linear(d,dm)
        self.lstm=nn.LSTM(dm,dm,num_layers=nl,batch_first=True,dropout=drop)
        self.ln1=nn.LayerNorm(dm)
        enc=nn.TransformerEncoderLayer(dm,nh,dm*2,drop,batch_first=True,norm_first=True)
        self.attn=nn.TransformerEncoder(enc,2)
        self.glu=GLU(dm); self.ln2=nn.LayerNorm(dm)
        self.fc=nn.Sequential(nn.Linear(dm,64),nn.GELU(),nn.Dropout(drop),nn.Linear(64,nc))
    def forward(self,x):
        w=self.vsn(x.mean(dim=1,keepdim=True)); x=x*w; x=self.proj(x)
        lo,_=self.lstm(x); lo=self.ln1(lo+x)
        ao=self.attn(lo); out=self.ln2(self.glu(ao)+ao)
        return self.fc(out[:,-1,:])

def compute_cw(y,nc=4):
    c=np.bincount(y,minlength=nc); w=1./(c+1e-6)
    return torch.tensor(w/w.sum()*nc,dtype=torch.float32)

def train_dl(model,dl_tr,dl_va,cw=None,epochs=CONFIG['dl_epochs'],lr=CONFIG['dl_lr'],label='',patience=8):
    crit=FocalLoss(gamma=2.,alpha=cw,ls=0.1)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,T_0=10,T_mult=2)
    best_loss,best_st,wait=float('inf'),None,0; t0=time.time()
    for ep in range(epochs):
        model.train()
        for bx,by in dl_tr:
            bx,by=bx.to(device),by.to(device); opt.zero_grad()
            loss=crit(model(bx),by); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step()
        sched.step()
        model.eval(); vl=0.
        with torch.no_grad():
            for bx,by in dl_va: bx,by=bx.to(device),by.to(device); vl+=crit(model(bx),by).item()
        if vl<best_loss: best_loss=vl; best_st={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: print(f"  [{label}] Stop ep{ep+1} ({time.time()-t0:.0f}s)"); break
        if (ep+1)%10==0: print(f"  [{label}] ep{ep+1} val_loss={vl:.4f} ({time.time()-t0:.0f}s)")
    if best_st: model.load_state_dict(best_st)

def eval_dl(model,loader):
    model.eval(); preds,probs,targets=[],[],[]
    with torch.no_grad():
        for bx,by in loader:
            logits=model(bx.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            probs.extend(torch.softmax(logits,dim=1).cpu().numpy())
            targets.extend(by.numpy())
    return np.array(targets),np.array(preds),np.array(probs)

print("Architecture DL définie: TFT (Temporal Fusion Transformer)")


## Validation walk-forward du TFT — méthodologie

Le rapport donne pour le **TFT (Temporal Fusion Transformer)** à h=5j : $F_1^{\text{UP\_FORT}}=0.588$ — la seule fois où le projet a franchi 0.50 sur les fortes hausses. Mais ce chiffre vient d'un **split statique unique**, exactement le biais qui a fait s'effondrer le champion STRESS en walk-forward. Cette notebook rejoue le TFT (architecture reprise **verbatim** de `VIX_ML3`, non modifiée) sur 5 folds walk-forward, comparé au RandomForest GLOBAL.

### Un ajustement méthodologique délibéré : pas de SMOTE avant fenêtrage

Dans le pipeline DL original, le suréchantillonnage (SMOTE/BorderlineSMOTE) était appliqué **avant** de découper les fenêtres glissantes de `dl_lookback=21` jours pour le TFT. C'est problématique pour un modèle séquentiel : une fenêtre de 21 lignes consécutives dans un tableau **rééchantillonné** n'est plus une fenêtre temporelle réelle — les lignes adjacentes après SMOTE ne sont plus des jours consécutifs. La fenêtre perd son sens.

**Ici** : aucun rééchantillonnage n'est appliqué avant le fenêtrage — les fenêtres du TFT restent de vraies séquences chronologiques de 21 jours. Le déséquilibre des classes est géré uniquement par la **Focal Loss pondérée par classe** (`compute_cw`, déjà présente dans l'architecture d'origine), ce qui est la bonne pratique standard pour les modèles séquentiels déséquilibrés (Lin et al., 2017).

### Hold-out chronologique réel pour l'early stopping
Comme pour la correction déjà appliquée dans `VIX_ML3` : le dernier 15% du train de chaque fold sert de validation pour l'arrêt anticipé — jamais de rééchantillonnage synthétique dans ce hold-out, et jamais vu pendant l'entraînement.

### Comparateurs
- **GLOBAL RandomForest N=8** (le même protocole que `VIX_CHAMPION_WF`/`VIX_REGIME_ROUTER`).
- **Ensemble TFT+RF** : moyenne simple des probabilités des deux modèles.


In [ ]:
# ============================================================
# SYNCHRONISATION DE LA PROGRESSION AVEC GITHUB
# [IMPORTANT] Le disque Colab est éphémère d'une session à l'autre (nouvelle
# VM à chaque reconnexion) : RESULTS_CSV local ne survit PAS à une
# déconnexion. Pour reprendre après une coupure (quasi certaine vu la durée
# du scan), il faut (1) récupérer la progression déjà poussée AVANT de
# recalculer done_keys, et (2) republier régulièrement PENDANT le run, pas
# seulement à la fin — sinon un plantage juste avant la fin ferait tout
# perdre.
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

_PUSH_WORKDIR = "/content/_vix_ml_scan_push"

def pull_progress():
    """Récupère RESULTS_CSV déjà poussé (sessions précédentes), s'il existe."""
    if os.path.exists(RESULTS_CSV):
        print(f"[SKIP PULL] {RESULTS_CSV} déjà présent localement.")
        return
    if not GITHUB_TOKEN:
        print("[SKIP PULL] Pas de GITHUB_TOKEN — impossible de vérifier une progression antérieure poussée. "
              "Le scan démarre de zéro dans ce runtime.")
        return
    url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    exists = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", url, RESULTS_BRANCH],
                            capture_output=True, text=True)
    if exists.returncode != 0:
        print(f"[INFO] Aucune progression antérieure sur '{RESULTS_BRANCH}' — nouveau run.")
        return
    workdir = "/content/_vix_ml_scan_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", RESULTS_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode == 0 and os.path.exists(f"{workdir}/{RESULTS_CSV}"):
        subprocess.run(["cp", f"{workdir}/{RESULTS_CSV}", "."], check=True)
        n = sum(1 for _ in open(RESULTS_CSV)) - 1
        print(f"[PULL OK] Progression antérieure récupérée : ~{n} lignes déjà faites.")
    else:
        print(f"[WARN] Branche '{RESULTS_BRANCH}' trouvée mais {RESULTS_CSV} absent — nouveau run.")

def push_progress(label=''):
    """Publie l'état courant de RESULTS_CSV (à appeler périodiquement + en fin de run)."""
    if not GITHUB_TOKEN or not os.path.exists(RESULTS_CSV):
        return False
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0: return False
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)
        subprocess.run(["cp", RESULTS_CSV, f"{_PUSH_WORKDIR}/{RESULTS_CSV}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email", "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name", "VIX Final ML Scan Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", RESULTS_CSV], check=True)
        commit = subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                                 f"Progression scan ML {label} — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                                capture_output=True, text=True)
        if 'nothing to commit' in (commit.stdout or ''):
            return True
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        ok = push.returncode == 0
        if ok: print(f"  [CHECKPOINT PUSHÉ] {label} ({pd.Timestamp.now():%H:%M:%S})")
        else: print(f"  [WARN push checkpoint] {push.stderr[-300:]}")
        return ok
    except Exception as e:
        print(f"  [WARN push checkpoint] {e}")
        return False

pull_progress()


In [ ]:
# ============================================================
# TFT SUR TOUS LES HORIZONS × RÉGIMES × FOLDS — CHECKPOINTÉ ET RÉSUMABLE
# Même principe anti-SMOTE-avant-fenêtrage que VIX_TFT_WF (fenêtres de 21j
# chronologiques réelles ; déséquilibre géré par la Focal Loss pondérée).
# ============================================================
KEY_COLS = ['horizon', 'regime', 'fold']
lb = CONFIG['dl_lookback']

done_keys = set()
if os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0:
    prev = pd.read_csv(RESULTS_CSV, usecols=KEY_COLS)
    done_keys = set(map(tuple, prev.values.tolist()))
    print(f"[REPRISE] {len(done_keys)}/{n_combos} modèles TFT déjà faits — reprise en cours.")
else:
    print("[DÉMARRAGE] Aucun résultat existant — nouveau run.")

def already_done(h, reg, fold):
    return (h, reg, fold) in done_keys

def save_row(row):
    header = not (os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=header, index=False)
    done_keys.add(tuple(row[c] for c in KEY_COLS))

dl_feats = [f for f in DL_FEATURE_SET if f in df_features.columns]
print(f"Features DL disponibles: {len(dl_feats)}/{len(DL_FEATURE_SET)}")

t0 = time.time(); n_done_session = 0; import gc

for h in CONFIG['horizons']:
    for reg in CONFIG['regimes']:
        for k in range(CONFIG['n_wf_folds']):
            if already_done(h, reg, k + 1):
                continue
            cut, nxt = FOLD_CUTS[k], FOLD_CUTS[k + 1]
            cut_date, nxt_date = all_dates[cut], all_dates[nxt - 1]
            target, reg_r, _ = build_target(df_features[VIX_COL], h, cut)
            idx = target.index
            reg_al = reg_r.reindex(idx).fillna('NORMAL').values
            tr_mask = np.asarray(idx < cut_date)
            te_mask = np.asarray((idx >= cut_date) & (idx <= nxt_date))
            if reg != 'GLOBAL':
                tr_mask = tr_mask & (reg_al == reg)
                te_mask = te_mask & (reg_al == reg)
            y_tr_all = target.values[tr_mask].astype(int); y_te = target.values[te_mask].astype(int)
            if len(y_te) < CONFIG['min_test_rows'] or len(y_tr_all) < CONFIG['min_train_rows']:
                save_row({'horizon': h, 'regime': reg, 'fold': k + 1, 'skipped': 'echantillon_insuffisant',
                          'F1_dir': np.nan, 'F1_UP_FORT': np.nan, 'F1_DOWN_FORT': np.nan})
                n_done_session += 1
                continue

            pos_cut = int(tr_mask.sum()); pos_nxt = pos_cut + int(te_mask.sum())
            X_dl_all = df_features[dl_feats].reindex(idx).fillna(0).values
            # Pour un régime != GLOBAL, tr_mask/te_mask ne sont pas contigus (jours
            # dispersés dans le temps) : le fenêtrage par position n'a de sens que
            # sur une série contiguë. On restreint donc X_dl_all/target à l'union
            # triée des positions du régime pour ce fold, en gardant l'ordre
            # temporel (fenêtre = 21 jours DU RÉGIME les plus récents, pas 21 jours
            # calendaires) — cohérent avec le principe déjà appliqué pour le ML.
            sel_pos = np.where(tr_mask | te_mask)[0]
            X_sel = X_dl_all[sel_pos]; y_sel = target.values[sel_pos].astype(int)
            n_tr_sel = int(tr_mask[sel_pos].sum())
            hold_sp = int(n_tr_sel * (1 - CONFIG['dl_holdout_frac']))
            if hold_sp < 100 or (n_tr_sel - hold_sp) < lb + 10 or n_tr_sel < lb:
                save_row({'horizon': h, 'regime': reg, 'fold': k + 1, 'skipped': 'trop_peu_pour_fenetrage',
                          'F1_dir': np.nan, 'F1_UP_FORT': np.nan, 'F1_DOWN_FORT': np.nan})
                n_done_session += 1
                continue

            sc_dl = RobustScaler().fit(X_sel[:hold_sp])
            X_core = sc_dl.transform(X_sel[:hold_sp]); y_core = y_sel[:hold_sp]
            X_hold = sc_dl.transform(X_sel[hold_sp:n_tr_sel]); y_hold = y_sel[hold_sp:n_tr_sel]
            X_test_win = sc_dl.transform(X_sel[n_tr_sel - lb:])
            y_test_win = y_sel[n_tr_sel - lb:]

            ds_tr = PreScaledDS(X_core, y_core, lb=lb)
            ds_va = PreScaledDS(X_hold, y_hold, lb=lb)
            ds_te = PreScaledDS(X_test_win, y_test_win, lb=lb)
            dl_tr = DataLoader(ds_tr, batch_size=CONFIG['dl_batch'], shuffle=True)
            dl_va = DataLoader(ds_va, batch_size=256)
            dl_te = DataLoader(ds_te, batch_size=256)

            cw = compute_cw(y_core)
            model = VIX_TFT(len(dl_feats)).to(device)
            train_dl(model, dl_tr, dl_va, cw=cw, label=f'TFT_h{h}_{reg}_f{k+1}', patience=8)
            y_te_tft, pred_tft, prob_tft = eval_dl(model, dl_te)
            met = metrics(y_te_tft, pred_tft)
            del model; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

            save_row({'horizon': h, 'regime': reg, 'fold': k + 1, 'skipped': '',
                      'n_train': n_tr_sel, 'n_test': len(y_te_tft),
                      'test_start': str(cut_date.date()), 'test_end': str(nxt_date.date()), **met})
            n_done_session += 1
            elapsed = time.time() - t0
            rate = n_done_session / elapsed
            eta_h = ((n_combos - len(done_keys)) / rate) / 3600 if rate > 0 else float('nan')
            print(f"  [{len(done_keys)}/{n_combos}] h={h}j {reg} fold{k+1} F1_dir={met['F1_dir']:.3f} "
                  f"UP={met['F1_UP_FORT']} | {elapsed/60:.1f}min écoulées, ETA restante≈{eta_h:.1f}h")
            if n_done_session % 10 == 0:
                push_progress(label=f"{len(done_keys)}/{n_combos}")

push_progress(label=f"fin de session ({len(done_keys)}/{n_combos})")
print(f"\n[TFT] {len(done_keys)}/{n_combos} modèles au total "
      f"({n_done_session} faits cette session, {(time.time()-t0)/60:.1f}min)")


In [ ]:
# ============================================================
# SYNTHÈSE (à date)
# ============================================================
df_tft = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
print(f"Progression: {len(df_tft)}/{n_combos} ({len(df_tft)/max(n_combos,1):.1%})")

if len(df_tft):
    ok = df_tft[df_tft['F1_dir'].notna()]
    agg = (ok.groupby(['horizon', 'regime'])
           .agg(F1_dir_mean=('F1_dir', 'mean'), F1_dir_std=('F1_dir', 'std'),
                F1_UP_FORT_mean=('F1_UP_FORT', 'mean'), F1_DOWN_FORT_mean=('F1_DOWN_FORT', 'mean'),
                n_folds=('fold', 'nunique'))
           .reset_index().round(4))
    print("\n### TFT — moyenne walk-forward par (horizon, régime) ###")
    print(agg.sort_values('F1_dir_mean', ascending=False).to_string(index=False))
    best_up = agg.sort_values('F1_UP_FORT_mean', ascending=False).head(5)
    print("\n### Meilleurs F1_UP_FORT ###")
    print(best_up.to_string(index=False))

    print("\nRéférences établies (walk-forward) :")
    print("  TFT h=5j (VIX_TFT_WF)        F1_dir=0.559±0.049  F1_UP_FORT=0.118  (infirmé)")
    print("  GLOBAL RandomForest h=5j     F1_dir=0.610±0.025  F1_UP_FORT=0.359")

    try:
        with pd.ExcelWriter('VIX_FINAL_TFT_report.xlsx', engine='xlsxwriter') as w:
            agg.sort_values('F1_dir_mean', ascending=False).to_excel(w, 'Agg_by_config', index=False)
            df_tft.to_excel(w, 'Detail', index=False)
        print("\n[SAVE] VIX_FINAL_TFT_report.xlsx (snapshot à date)")
    except Exception as e:
        print(f"[WARN Export] {e}")
else:
    print("Aucun résultat pour l'instant.")
print(f"\n[NOTE] {RESULTS_CSV} contient le détail complet — le recharger pour reprendre.")


In [ ]:
# ============================================================
# PUSH FINAL DU RAPPORT (xlsx) EN PLUS DU CSV DE PROGRESSION
# ============================================================
def push_report_file():
    if not GITHUB_TOKEN or not os.path.exists('VIX_FINAL_TFT_report.xlsx'):
        print("[SKIP] Pas de token ou pas de rapport à pousser.")
        return
    try:
        subprocess.run(["cp", "VIX_FINAL_TFT_report.xlsx", f"{_PUSH_WORKDIR}/VIX_FINAL_TFT_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", "VIX_FINAL_TFT_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Rapport TFT agrégé — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] VIX_FINAL_TFT_report.xlsx sur '{RESULTS_BRANCH}'")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_progress(label='rapport final')
push_report_file()
